# Fine-Tune Qwen 2.5 Coder (3B) on T4 GPU (Optimized)

This notebook covers fine-tuning on a **Tesla T4 GPU** (Standard Colab).

### ⚠️ Configuration
This script is configured for **Speed** and **Stability**:
- **4-bit Quantization**: To fit in 16GB VRAM.
- **Float16 Compute**: For faster training than FP32.
- **AMP Disabled**: To preventing crashing on T4 GPUs (BFloat16 error).

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
# ------------------------------------------------------------------
# 0. CRITICAL: Force Disable Mixed Precision in Environment
# This prevents the 'GradScaler' from running and crashing on BF16 tensors
# ------------------------------------------------------------------
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Upload Dataset

Please upload the `sql_to_mql_finetuning_short_version.jsonl` file.

In [ ]:
from google.colab import files
import os

if not os.path.exists('sql_to_mql_finetuning_short_version.jsonl'):
    print("Please upload 'sql_to_mql_finetuning_short_version.jsonl'.")
    uploaded = files.upload()

## 3. Fine-Tuning Script

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
import gc

# 1. Clean Slate
torch.cuda.empty_cache()
gc.collect()

# 2. Config
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
DATA_PATH = "sql_to_mql_finetuning_short_version.jsonl"
NEW_MODEL = "Qwen2.5-Coder-3B-Instruct-mql-adapter"

# 3. Quantization (Optimized for Speed)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # <--- FAST Compute (FP16)
    bnb_4bit_use_double_quant=False,
)

# 4. Load Model 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16 # <--- Load native weights in FP16
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

base_model = prepare_model_for_kbit_training(base_model)

# 5. LoRA Config
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
base_model = get_peft_model(base_model, peft_config)
base_model.print_trainable_parameters()

# 6. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 7. Data
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_chat_template(row):
    conversation = row["messages"]
    text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat_template)
print(f"Dataset Size: {len(dataset)}")

# 8. SFT Config (Balanced)
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8, 
    optim="paged_adamw_8bit",
    save_steps=100,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,     # <--- DISABLED to avoid Scaler crash
    bf16=False,     # <--- DISABLED
    max_grad_norm=0.3,
    group_by_length=True,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

# 9. Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting Training...")
trainer.train()

print("Saving model...")
trainer.model.save_pretrained(NEW_MODEL)
tokenizer.save_pretrained(NEW_MODEL)
print("Training Complete")

## 4. Save to Drive

Mount Google Drive and save the adapter for persistent use.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest_path = f"/content/drive/MyDrive/{NEW_MODEL}"
shutil.copytree(NEW_MODEL, dest_path)
print(f"Model saved to {dest_path}")